# 10年定着予測 - 系統の異なるモデルの検証（NN(MLP+Embedding) vs 非線形SVM、現行441列）

**背景**: `13_autogluon_prototype`で、NeuralNetTorch(val 0.5997)・NeuralNetFastAI(val 0.6025)は
AutoGluonの全11モデル中最下位だった（CatBoost 0.5679が最良）。ただしこれはAutoGluonの
**ゼロショット既定設定**でのNN比較であり、丁寧なチューニングはされていない。`31_`のL2ロジスティック
回帰（線形モデル）はCatBoostに大きく劣った（val 0.57台 vs 0.50台）ことも踏まえ、
**「決定木ではないが、非線形性を捉えられるモデル」がCatBoostに近づけるか**を検証する。

## 検証するモデル

1. **NN(MLP+Embedding)**: カテゴリ変数をEmbedding層で低次元ベクトル化し、数値変数
   （中央値補完＋欠損フラグ＋標準化、`31_`と同様の前処理）と連結してMLPに入力する。
   Optunaで層のサイズ・dropout率・学習率等をチューニングする。
2. **非線形SVM(RBFカーネル)**: `31_`と同じOne-Hot＋標準化した特徴量行列に対し、
   `sklearn.svm.SVC(kernel="rbf", probability=True)`をOptunaでC・gammaをチューニングする。

CatBoost（`28_`と同一構成）を参考の比較対象として同時に再学習する。

## 検証方法

80/20・75/25の2つの時系列splitで、CatBoost（参考）・NN・SVMを比較する。

## 実行環境
Google Colab（CPU）を想定。NN・SVMはCatBoostよりモデル自体は軽量だが、Optuna探索の
試行数はCPU実行時間を踏まえて15回に抑える（GBDT系の25回より少ない）。


In [9]:
!pip install -q catboost optuna torch

In [10]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [11]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import datetime
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
import torch
import torch.nn as nn
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
torch.set_num_threads(4)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [13]:
SCRIPT_NAME = "36_alternative_models"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-11 12:39:28] [INFO] === [36_alternative_models] 実験開始 ===


INFO:36_alternative_models:=== [36_alternative_models] 実験開始 ===


[2026-08-11 12:39:28] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260811


INFO:36_alternative_models:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260811


[2026-08-11 12:39:28] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/36_alternative_models_checkpoint.csv


INFO:36_alternative_models:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/36_alternative_models_checkpoint.csv


[2026-08-11 12:39:28] [INFO] チェックポイントは未作成（新規実行）


INFO:36_alternative_models:チェックポイントは未作成（新規実行）


In [14]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-11 12:39:29] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:36_alternative_models:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-11 12:39:29] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:36_alternative_models:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-11 12:39:29] [INFO] 定着率: 0.5647


INFO:36_alternative_models:定着率: 0.5647


[2026-08-11 12:39:29] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:36_alternative_models:Train IDs: 2761, Test IDs: 2502


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [15]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [16]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-11 12:39:29] [INFO] ------------------------------------------------------------


INFO:36_alternative_models:------------------------------------------------------------


[2026-08-11 12:39:29] [INFO] split非依存の基本特徴量を生成中...


INFO:36_alternative_models:split非依存の基本特徴量を生成中...


[2026-08-11 12:39:29] [INFO] ------------------------------------------------------------


INFO:36_alternative_models:------------------------------------------------------------


[2026-08-11 12:46:59] [INFO] split非依存の基本特徴量生成完了


INFO:36_alternative_models:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [17]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-11 12:46:59] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:36_alternative_models:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-11 12:47:01] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:36_alternative_models:入社時メモ: SVD累積寄与率=0.760


[2026-08-11 12:47:06] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:36_alternative_models:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-11 12:47:08] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:36_alternative_models:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-11 12:47:08] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:36_alternative_models:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [18]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-11 12:47:09] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:36_alternative_models:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-11 12:50:00] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:36_alternative_models:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [19]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-11 12:50:00] [INFO] Persona単位の基本特徴量を生成中...


INFO:36_alternative_models:Persona単位の基本特徴量を生成中...


[2026-08-11 12:50:00] [INFO] Persona単位の基本特徴量処理完了


INFO:36_alternative_models:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版）

`転居許容`フラグの抽出ロジックは`27_`と同一。`希望勤務地`の抽出のみ2種類を用意する：

- **v1**: `27_`・`25_`・`data_exploration_v3/v4/v5`と同一の正規表現（Public 0.529672で確認済み）
- **v2**: v1に加え、「◯◯を希望。」「◯◯勤務を希望。」「◯◯での勤務を希望。」パターンを追加で
  拾う拡張版。未抽出だった152件（train）を目視確認して発見した言い回し。カバー率が
  88.6%→94.1%（train）/ 95.0%（test）に向上し、ダブル悪条件の該当件数も342→354件に増加、
  効果量はp=1.4×10⁻³⁹→1.7×10⁻⁴³・オッズ比0.189→0.178とむしろ強まった。

In [20]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())

[2026-08-11 12:50:01] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:36_alternative_models:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-11 12:50:01] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:36_alternative_models:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-11 12:50:01] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:36_alternative_models:L_v2: Train (2761, 3), Test (2502, 3)


L_v1 ダブル悪条件:
転居x勤務地_ダブル悪条件_v1
0    2419
1     342
Name: count, dtype: int64

L_v2 ダブル悪条件:
転居x勤務地_ダブル悪条件_v2
0    2407
1     354
Name: count, dtype: int64


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"L1"}`/`{"L2"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の構成、Eなし）に対してL_v1・L_v2のいずれかを単体で追加できるようにする。

In [21]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None):
    '''指定した分割比率で特徴量を組み立てる。extra_blocks: {"L1","L2"}のサブセット（通常はどちらか一方）'''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了")

✅ 部署Target Encoding・prepare_split関数定義完了


## 7. チェックポイント機能（`18_`〜`27_`と同一）

In [22]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=["config", "n_features", "val_score", "submission_path"])

def save_checkpoint_row(result):
    df = pd.DataFrame([result])
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']:.6f}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了")

✅ チェックポイント関数定義完了


## 8. モデル実行関数（CatBoost・NN(MLP+Embedding)・SVM(RBF)）

CatBoostは参考の比較対象として`28_`と同一構成で再学習する。NN・SVMは`31_`と同様に
One-Hot・標準化・中央値補完（欠損フラグ付き）した特徴量行列を使う（NNはカテゴリ変数を
Embedding層で扱うため、カテゴリ変数はOne-Hotではなく整数インデックスにエンコードする）。

In [23]:
def _save_and_score(val_preds, test_preds, y_va, test_features, config_label, n_features):
    val_score = log_loss(y_va, val_preds)
    sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    sub = pd.DataFrame({ID_COL: test_features.index, TARGET_COL: test_preds})
    sub.to_csv(sub_path, index=False, header=False)
    val_pred_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_valpreds.npy"
    np.save(val_pred_path, val_preds)
    logger.info(f"[{config_label}] n_features={n_features}, val_score={val_score:.6f}")
    return {
        "config": config_label, "n_features": n_features, "val_score": val_score,
        "submission_path": str(sub_path), "val_pred_path": str(val_pred_path),
    }


def run_model_config_catboost(ag_train_data, ag_tuning_data, test_features, config_label, n_trials=25):
    feature_cols = [c for c in ag_train_data.columns if c not in ["入社日", TARGET_COL]]
    obj_cols = [c for c in feature_cols if ag_train_data[c].dtype == "object"]

    X_tr = ag_train_data[feature_cols].fillna(-999)
    y_tr = ag_train_data[TARGET_COL]
    X_va = ag_tuning_data[feature_cols].fillna(-999)
    y_va = ag_tuning_data[TARGET_COL]
    X_test = test_features[feature_cols].fillna(-999)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
            "thread_count": 4,
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params

    final_model = cb.CatBoostClassifier(
        **best_params, iterations=3000, random_seed=SEED, verbose=False,
        cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU", thread_count=4,
    )
    final_model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
    val_preds = final_model.predict_proba(X_va)[:, 1]
    test_preds = final_model.predict_proba(X_test)[:, 1]
    return _save_and_score(val_preds, test_preds, y_va, test_features, config_label, len(feature_cols))


def _prepare_onehot_features(ag_train_data, ag_tuning_data, test_features, feature_cols, obj_cols, num_cols):
    """31_と同様のOne-Hot・標準化・中央値補完（欠損フラグ付き）前処理"""
    X_tr_raw = ag_train_data[feature_cols].copy()
    X_va_raw = ag_tuning_data[feature_cols].copy()
    X_test_raw = test_features[feature_cols].copy()

    missing_flag_cols = [
        c for c in num_cols
        if X_tr_raw[c].isna().any() or X_va_raw[c].isna().any() or X_test_raw[c].isna().any()
    ]
    for c in missing_flag_cols:
        X_tr_raw[f"{c}_missing"] = X_tr_raw[c].isna().astype(float)
        X_va_raw[f"{c}_missing"] = X_va_raw[c].isna().astype(float)
        X_test_raw[f"{c}_missing"] = X_test_raw[c].isna().astype(float)

    medians = X_tr_raw[num_cols].median()
    X_tr_num = X_tr_raw[num_cols].fillna(medians).fillna(0.0)
    X_va_num = X_va_raw[num_cols].fillna(medians).fillna(0.0)
    X_test_num = X_test_raw[num_cols].fillna(medians).fillna(0.0)

    scaler = StandardScaler()
    X_tr_num_scaled = np.clip(scaler.fit_transform(X_tr_num), -10, 10)
    X_va_num_scaled = np.clip(scaler.transform(X_va_num), -10, 10)
    X_test_num_scaled = np.clip(scaler.transform(X_test_num), -10, 10)

    X_tr_cat = X_tr_raw[obj_cols].fillna("missing").astype(str)
    X_va_cat = X_va_raw[obj_cols].fillna("missing").astype(str)
    X_test_cat = X_test_raw[obj_cols].fillna("missing").astype(str)

    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    X_tr_cat_enc = ohe.fit_transform(X_tr_cat)
    X_va_cat_enc = ohe.transform(X_va_cat)
    X_test_cat_enc = ohe.transform(X_test_cat)

    if missing_flag_cols:
        mflag_cols = [f"{c}_missing" for c in missing_flag_cols]
        mflags_tr, mflags_va, mflags_test = X_tr_raw[mflag_cols].values, X_va_raw[mflag_cols].values, X_test_raw[mflag_cols].values
    else:
        mflags_tr = np.zeros((len(X_tr_raw), 0))
        mflags_va = np.zeros((len(X_va_raw), 0))
        mflags_test = np.zeros((len(X_test_raw), 0))

    X_tr_final = np.hstack([X_tr_num_scaled, X_tr_cat_enc, mflags_tr]).astype(np.float32)
    X_va_final = np.hstack([X_va_num_scaled, X_va_cat_enc, mflags_va]).astype(np.float32)
    X_test_final = np.hstack([X_test_num_scaled, X_test_cat_enc, mflags_test]).astype(np.float32)
    return X_tr_final, X_va_final, X_test_final


def run_model_config_svm(ag_train_data, ag_tuning_data, test_features, config_label, n_trials=15):
    feature_cols = [c for c in ag_train_data.columns if c not in ["入社日", TARGET_COL]]
    obj_cols = [c for c in feature_cols if ag_train_data[c].dtype == "object"]
    num_cols = [c for c in feature_cols if c not in obj_cols]
    y_tr = ag_train_data[TARGET_COL].values
    y_va = ag_tuning_data[TARGET_COL].values

    X_tr_final, X_va_final, X_test_final = _prepare_onehot_features(
        ag_train_data, ag_tuning_data, test_features, feature_cols, obj_cols, num_cols
    )

    def objective(trial):
        C = trial.suggest_float("C", 1e-2, 1e2, log=True)
        gamma = trial.suggest_float("gamma", 1e-5, 1e-1, log=True)
        model = SVC(kernel="rbf", C=C, gamma=gamma, probability=True, random_state=SEED)
        model.fit(X_tr_final, y_tr)
        return log_loss(y_va, model.predict_proba(X_va_final)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params

    final_model = SVC(kernel="rbf", **best_params, probability=True, random_state=SEED)
    final_model.fit(X_tr_final, y_tr)
    val_preds = final_model.predict_proba(X_va_final)[:, 1]
    test_preds = final_model.predict_proba(X_test_final)[:, 1]
    logger.info(f"[{config_label}] best_C={best_params['C']:.4g}, best_gamma={best_params['gamma']:.4g}")
    return _save_and_score(val_preds, test_preds, ag_tuning_data[TARGET_COL], test_features, config_label, X_tr_final.shape[1])


class EmbeddingMLP(nn.Module):
    def __init__(self, cat_cardinalities, num_numeric, hidden1=128, hidden2=64, dropout=0.3):
        super().__init__()
        emb_dims = [min(50, (card + 2) // 2) for card in cat_cardinalities]
        self.embeddings = nn.ModuleList([nn.Embedding(card + 1, dim) for card, dim in zip(cat_cardinalities, emb_dims)])
        input_dim = sum(emb_dims) + num_numeric
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.BatchNorm1d(hidden1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2), nn.BatchNorm1d(hidden2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
        )

    def forward(self, cat_x, num_x):
        embs = [emb(cat_x[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(embs + [num_x], dim=1)
        return self.mlp(x).squeeze(-1)


def _prepare_nn_features(ag_train_data, ag_tuning_data, test_features, feature_cols, obj_cols, num_cols):
    """カテゴリ変数は整数インデックス(Embedding用)、数値は標準化(One-Hotなし)"""
    X_tr_raw = ag_train_data[feature_cols].copy()
    X_va_raw = ag_tuning_data[feature_cols].copy()
    X_test_raw = test_features[feature_cols].copy()

    missing_flag_cols = [
        c for c in num_cols
        if X_tr_raw[c].isna().any() or X_va_raw[c].isna().any() or X_test_raw[c].isna().any()
    ]
    for c in missing_flag_cols:
        X_tr_raw[f"{c}_missing"] = X_tr_raw[c].isna().astype(float)
        X_va_raw[f"{c}_missing"] = X_va_raw[c].isna().astype(float)
        X_test_raw[f"{c}_missing"] = X_test_raw[c].isna().astype(float)
    num_cols_final = num_cols + [f"{c}_missing" for c in missing_flag_cols]

    medians = X_tr_raw[num_cols].median()
    X_tr_num = X_tr_raw[num_cols].fillna(medians).fillna(0.0)
    X_va_num = X_va_raw[num_cols].fillna(medians).fillna(0.0)
    X_test_num = X_test_raw[num_cols].fillna(medians).fillna(0.0)
    scaler = StandardScaler()
    X_tr_num_scaled = np.clip(scaler.fit_transform(X_tr_num), -10, 10)
    X_va_num_scaled = np.clip(scaler.transform(X_va_num), -10, 10)
    X_test_num_scaled = np.clip(scaler.transform(X_test_num), -10, 10)
    if missing_flag_cols:
        mflag_cols = [f"{c}_missing" for c in missing_flag_cols]
        X_tr_num_scaled = np.hstack([X_tr_num_scaled, X_tr_raw[mflag_cols].values])
        X_va_num_scaled = np.hstack([X_va_num_scaled, X_va_raw[mflag_cols].values])
        X_test_num_scaled = np.hstack([X_test_num_scaled, X_test_raw[mflag_cols].values])

    cat_cardinalities = []
    X_tr_cat_idx, X_va_cat_idx, X_test_cat_idx = [], [], []
    for c in obj_cols:
        vocab = {v: i for i, v in enumerate(X_tr_raw[c].fillna("missing").astype(str).unique())}
        unk_idx = len(vocab)
        cat_cardinalities.append(len(vocab) + 1)
        X_tr_cat_idx.append(X_tr_raw[c].fillna("missing").astype(str).map(vocab).fillna(unk_idx).astype(int).values)
        X_va_cat_idx.append(X_va_raw[c].fillna("missing").astype(str).map(vocab).fillna(unk_idx).astype(int).values)
        X_test_cat_idx.append(X_test_raw[c].fillna("missing").astype(str).map(vocab).fillna(unk_idx).astype(int).values)

    X_tr_cat_idx = np.stack(X_tr_cat_idx, axis=1) if obj_cols else np.zeros((len(X_tr_raw), 0), dtype=int)
    X_va_cat_idx = np.stack(X_va_cat_idx, axis=1) if obj_cols else np.zeros((len(X_va_raw), 0), dtype=int)
    X_test_cat_idx = np.stack(X_test_cat_idx, axis=1) if obj_cols else np.zeros((len(X_test_raw), 0), dtype=int)

    return (
        X_tr_cat_idx, X_tr_num_scaled.astype(np.float32),
        X_va_cat_idx, X_va_num_scaled.astype(np.float32),
        X_test_cat_idx, X_test_num_scaled.astype(np.float32),
        cat_cardinalities,
    )


def _train_nn(cat_cardinalities, num_numeric, X_tr_cat, X_tr_num, y_tr, X_va_cat, X_va_num, y_va,
              hidden1, hidden2, dropout, lr, weight_decay, batch_size, max_epochs=150, patience=15):
    torch.manual_seed(SEED)
    device = torch.device("cpu")
    model = EmbeddingMLP(cat_cardinalities, num_numeric, hidden1, hidden2, dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()

    X_tr_cat_t = torch.tensor(X_tr_cat, dtype=torch.long)
    X_tr_num_t = torch.tensor(X_tr_num, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
    X_va_cat_t = torch.tensor(X_va_cat, dtype=torch.long)
    X_va_num_t = torch.tensor(X_va_num, dtype=torch.float32)

    n = X_tr_cat_t.shape[0]
    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            if idx.shape[0] < 2:
                continue  # BatchNorm1dは学習モードでバッチサイズ1を許容しないため端数バッチを飛ばす
            optimizer.zero_grad()
            logits = model(X_tr_cat_t[idx], X_tr_num_t[idx])
            loss = loss_fn(logits, y_tr_t[idx])
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_va_cat_t, X_va_num_t)
            val_preds = torch.sigmoid(val_logits).numpy()
            val_loss = log_loss(y_va, val_preds)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_val_loss


def run_model_config_nn(ag_train_data, ag_tuning_data, test_features, config_label, n_trials=15):
    feature_cols = [c for c in ag_train_data.columns if c not in ["入社日", TARGET_COL]]
    obj_cols = [c for c in feature_cols if ag_train_data[c].dtype == "object"]
    num_cols = [c for c in feature_cols if c not in obj_cols]
    y_tr = ag_train_data[TARGET_COL].values
    y_va = ag_tuning_data[TARGET_COL].values

    (X_tr_cat, X_tr_num, X_va_cat, X_va_num, X_test_cat, X_test_num,
     cat_cardinalities) = _prepare_nn_features(ag_train_data, ag_tuning_data, test_features, feature_cols, obj_cols, num_cols)
    num_numeric = X_tr_num.shape[1]

    def objective(trial):
        hidden1 = trial.suggest_categorical("hidden1", [64, 128, 256])
        hidden2 = trial.suggest_categorical("hidden2", [32, 64, 128])
        dropout = trial.suggest_float("dropout", 0.1, 0.5)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
        _, val_loss = _train_nn(
            cat_cardinalities, num_numeric, X_tr_cat, X_tr_num, y_tr, X_va_cat, X_va_num, y_va,
            hidden1, hidden2, dropout, lr, weight_decay, batch_size,
        )
        return val_loss

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params

    final_model, final_val_loss = _train_nn(
        cat_cardinalities, num_numeric, X_tr_cat, X_tr_num, y_tr, X_va_cat, X_va_num, y_va,
        best_params["hidden1"], best_params["hidden2"], best_params["dropout"],
        best_params["lr"], best_params["weight_decay"], best_params["batch_size"],
        max_epochs=300, patience=25,
    )
    final_model.eval()
    with torch.no_grad():
        val_preds = torch.sigmoid(final_model(torch.tensor(X_va_cat, dtype=torch.long), torch.tensor(X_va_num, dtype=torch.float32))).numpy()
        test_preds = torch.sigmoid(final_model(torch.tensor(X_test_cat, dtype=torch.long), torch.tensor(X_test_num, dtype=torch.float32))).numpy()

    logger.info(f"[{config_label}] best_params={best_params}")
    return _save_and_score(val_preds, test_preds, ag_tuning_data[TARGET_COL], test_features, config_label, num_numeric + len(cat_cardinalities))


print("✅ CatBoost/NN(MLP+Embedding)/SVM(RBF)実行関数定義完了")

✅ CatBoost/NN(MLP+Embedding)/SVM(RBF)実行関数定義完了


## 9. モデル比較: CatBoost(参考) vs NN(MLP+Embedding) vs SVM(RBF)（441列、`28_`と同一特徴量）× 2 split

NN・SVMのOptunaは`n_trials=15`（GBDT系の25より少なめ、CPU実行時間を踏まえて）。

In [24]:
SPLIT_RATIOS = {"split_80_20": 0.8, "split_75_25": 0.75}
MODEL_RUNNERS = {
    "catboost": (run_model_config_catboost, 25),
    "nn_embedding": (run_model_config_nn, 15),
    "svm_rbf": (run_model_config_svm, 15),
}

model_comparison_results = []
for split_name, ratio in SPLIT_RATIOS.items():
    ag_train_data, ag_tuning_data, test_features_full = prepare_split(ratio, extra_blocks={"L2"})
    for model_name, (runner, n_trials) in MODEL_RUNNERS.items():
        config_label = f"{split_name}_{model_name}"
        def _run(runner=runner, n_trials=n_trials, ag_train_data=ag_train_data, ag_tuning_data=ag_tuning_data,
                 test_features_full=test_features_full, config_label=config_label):
            logger.info(f"=== {config_label} ===")
            return runner(ag_train_data, ag_tuning_data, test_features_full, config_label, n_trials=n_trials)
        result = run_or_resume(config_label, _run)
        result["split"] = split_name
        result["model"] = model_name
        model_comparison_results.append(result)

comparison_df = pd.DataFrame(model_comparison_results)
comparison_pivot = comparison_df.pivot(index="model", columns="split", values="val_score")
comparison_pivot["mean"] = comparison_pivot[["split_80_20", "split_75_25"]].mean(axis=1)
comparison_pivot["std"] = comparison_pivot[["split_80_20", "split_75_25"]].std(axis=1)
comparison_pivot = comparison_pivot.reindex(["catboost", "nn_embedding", "svm_rbf"])
comparison_pivot["mean_diff_vs_catboost"] = comparison_pivot["mean"] - comparison_pivot.loc["catboost", "mean"]
comparison_pivot = comparison_pivot.sort_values("mean")

logger.info("=" * 60)
logger.info("モデル比較結果（CatBoost vs NN(MLP+Embedding) vs SVM(RBF)、441列）")
logger.info("=" * 60)
logger.info("\n" + comparison_pivot.to_string())
print("\n■ モデル比較結果:")
print(comparison_pivot.to_string())
print("\n※ catboostのval_scoreは28_のColab実測値(split_80_20=0.503065)と近い水準になるはず")
print("※ (参考)13_のAutoGluon既定設定NNの検証値: NeuralNetTorch 0.5997 / NeuralNetFastAI 0.6025（CatBoost 0.5679）")

[2026-08-11 12:50:02] [INFO] === split_80_20_catboost ===


INFO:36_alternative_models:=== split_80_20_catboost ===


[2026-08-11 12:53:46] [INFO] [split_80_20_catboost] n_features=441, val_score=0.503065


INFO:36_alternative_models:[split_80_20_catboost] n_features=441, val_score=0.503065


[2026-08-11 12:53:46] [INFO] === split_80_20_nn_embedding ===


...

[2026-08-11 13:02:02] [INFO] ============================================================


INFO:36_alternative_models:============================================================


[2026-08-11 13:02:02] [INFO] 
split         split_75_25  split_80_20      mean       std  mean_diff_vs_catboost
model                                                                            
catboost         0.510125     0.503065  0.506595  0.004992               0.000000
nn_embedding     0.556311     0.539163  0.547737  0.012125               0.041142
svm_rbf          0.561714     0.541683  0.551699  0.014164               0.045103


INFO:36_alternative_models:
split         split_75_25  split_80_20      mean       std  mean_diff_vs_catboost
model                                                                            
catboost         0.510125     0.503065  0.506595  0.004992               0.000000
nn_embedding     0.556311     0.539163  0.547737  0.012125               0.041142
svm_rbf          0.561714     0.541683  0.551699  0.014164               0.045103



■ モデル比較結果:
split         split_75_25  split_80_20      mean       std  mean_diff_vs_catboost
model                                                                            
catboost         0.510125     0.503065  0.506595  0.004992               0.000000
nn_embedding     0.556311     0.539163  0.547737  0.012125               0.041142
svm_rbf          0.561714     0.541683  0.551699  0.014164               0.045103

※ catboostのval_scoreは28_のColab実測値(split_80_20=0.503065)と近い水準になるはず
※ (参考)13_のAutoGluon既定設定NNの検証値: NeuralNetTorch 0.5997 / NeuralNetFastAI 0.6025（CatBoost 0.5679）


## 10. 総合結果・提出候補

split_80_20における各モデルの結果を一覧化する。**NN・SVMのいずれかがCatBoostを明確に
上回った場合のみ提出候補とする**。

In [25]:
split_80_20_rows = comparison_df[comparison_df["split"] == "split_80_20"].set_index("model")
summary_rows = split_80_20_rows[["val_score", "submission_path"]].reindex(
    ["catboost", "nn_embedding", "svm_rbf"]
).reset_index()

logger.info("=" * 60)
logger.info("総合結果（split_80_20、提出候補一覧）")
logger.info("=" * 60)
logger.info("\n" + summary_rows.to_string())
print("\n■ 総合結果（split_80_20、提出候補一覧）:")
print(summary_rows.to_string(index=False))
print(f"\n(参考) 28_ CatBoost+L_v2: Public 0.529454（現時点の最良）")

summary_rows

[2026-08-11 13:02:02] [INFO] ============================================================


INFO:36_alternative_models:============================================================


[2026-08-11 13:02:02] [INFO] 総合結果（split_80_20、提出候補一覧）


INFO:36_alternative_models:総合結果（split_80_20、提出候補一覧）


[2026-08-11 13:02:02] [INFO] ============================================================


INFO:36_alternative_models:============================================================


[2026-08-11 13:02:02] [INFO] 
          model  val_score                                                                                                      submission_path
0      catboost   0.503065      /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_36_alternative_models_split_80_20_catboost.csv
1  nn_embedding   0.539163  /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_36_alternative_models_split_80_20_nn_embedding.csv
2       svm_rbf   0.541683       /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_36_alternative_models_split_80_20_svm_rbf.csv


INFO:36_alternative_models:
          model  val_score                                                                                                      submission_path
0      catboost   0.503065      /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_36_alternative_models_split_80_20_catboost.csv
1  nn_embedding   0.539163  /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_36_alternative_models_split_80_20_nn_embedding.csv
2       svm_rbf   0.541683       /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_36_alternative_models_split_80_20_svm_rbf.csv



■ 総合結果（split_80_20、提出候補一覧）:
       model  val_score                                                                                                     submission_path
    catboost   0.503065     /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_36_alternative_models_split_80_20_catboost.csv
nn_embedding   0.539163 /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_36_alternative_models_split_80_20_nn_embedding.csv
     svm_rbf   0.541683      /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_36_alternative_models_split_80_20_svm_rbf.csv

(参考) 28_ CatBoost+L_v2: Public 0.529454（現時点の最良）


,model,val_score,submission_path
0,catboost,0.503065,/content/drive/MyDrive/jaggle_2026/data/output...
1,nn_embedding,0.539163,/content/drive/MyDrive/jaggle_2026/data/output...
2,svm_rbf,0.541683,/content/drive/MyDrive/jaggle_2026/data/output...


## 11. まとめ・次のアクション

1. モデル比較結果（9節）で、`13_`のAutoGluon既定設定NN（val 0.5997〜0.6025、全11モデル中最下位）
   より、丁寧にチューニングしたNN(MLP+Embedding)が改善しているか確認する。SVMも同様に確認する。
2. NN・SVMのいずれかがCatBoostを明確に上回った場合のみ、そのモデルの`split_80_20`
   提出ファイルをKaggleに提出しPublicスコアを確認する。
3. CatBoost優位が再現された場合は、この方向の検証をここで打ち切り、
   **現時点の最良は引き続き28_ L_v2_extended（Public 0.529454）**とする。
4. 結果が出たら`data/output/submit_result_report.md`に追記し、`best_submission_status.md`も更新する。

### バックログ（今回は着手しない）
- NNの層構成・Optuna試行数を増やしたさらなるチューニング（もし有望な兆候があれば）
- SVMの前処理バリエーション（PCA次元圧縮等、高次元One-Hotに対するRBFカーネルの効きにくさへの対策）
